# Stage 4 - Verification

Checks the DATA, not the models. Run before the detector evaluation: a broken item counts as a detector error, so an unverified benchmark punishes detectors for being right.

One batched call per item. The judge sees the corrupted source plus every fact from the summary, targeted and retained together and shuffled, and returns a verdict per line.

| Pipeline | Targeted facts must be | Retained facts must be |
|---|---|---|
| Deletion | NOT_MENTIONED | SUPPORTED |
| Alteration | CONTRADICTED | SUPPORTED |

A targeted fact returning the other pipeline's verdict means the wrong edit was applied, and that is reported separately from a plain failure.

**Judge is `anthropic/claude-haiku-4.5`** - outside every generator and every detector, so the benchmark is not circular. Reasoning is disabled: the probe confirmed 0 reasoning tokens, and they bill at the completion rate.

Naturalness is deliberately NOT run here. It is triage for human review and does not gate accept or reject, so it runs later over the accepted set only, which is smaller and cheaper.

**Projected cost: 400 calls, about $0.43.**

In [1]:
import os
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath("source_corruption"))
import verification as V

load_dotenv(os.path.abspath("../../.env"))
api_keys = [k for k in (os.getenv("OPENROUTER_API_KEY_NEW"), os.getenv("OPENROUTER_API_KEY")) if k]
if not api_keys:
    raise ValueError("No API key found! Check the .env file at the repository root.")

BACKUP_FILE = "backup_verification.csv"
OUTPUT_FILE = "verification_results.csv"
CHECKPOINT_EVERY = 20
MAX_WORKERS = 4

print(f"{len(api_keys)} key(s). Judge: {V.JUDGE_MODEL}")

2 key(s). Judge: anthropic/claude-haiku-4.5


In [2]:
def split_facts(blob):
    """The CSV joins facts with ' | '. Rebuild them as (index, '', text) tuples."""
    if pd.isna(blob):
        return []
    return [(i, "", s.strip())
            for i, s in enumerate(str(blob).split(" | "), start=1) if s.strip()]


frames = []
for gen in ("luna", "deepseek"):
    for pipeline in ("deletion", "alteration"):
        f = pd.read_csv(f"pilot_{gen}_{pipeline}.csv")
        f["gen"] = gen
        f["pipeline"] = pipeline
        f["operation"] = "delete" if pipeline == "deletion" else "alter"
        if "n" not in f.columns:
            f["n"] = 1
        frames.append(f)
items = pd.concat(frames, ignore_index=True)

jobs = []
for idx, row in items.iterrows():
    targets = split_facts(row["target_facts"])
    retained = split_facts(row["retained_facts"])
    if not targets:
        continue
    prompt, roles = V.build_check(row["source_corrupted"], targets, retained, row["doc_id"])
    jobs.append({
        "job_key": f"{row['gen']}|{row['pipeline']}|{row['doc_id']}|{row['method']}|{row['n']}",
        "gen": row["gen"], "pipeline": row["pipeline"], "operation": row["operation"],
        "doc_id": row["doc_id"], "method": row["method"], "n": row["n"],
        "target_categories": row["target_categories"],
        "n_facts": len(roles), "roles": roles, "prompt": prompt,
    })

assert len({j["job_key"] for j in jobs}) == len(jobs), "duplicate job_key - would be double-paid"
print(f"{len(items)} items -> {len(jobs)} verification calls")
print(f"facts per call: mean {sum(j['n_facts'] for j in jobs) / len(jobs):.1f}, "
      f"max {max(j['n_facts'] for j in jobs)}")

400 items -> 400 verification calls
facts per call: mean 6.0, max 12


In [3]:
key_lock = threading.Lock()
active_key_index = 0


def call_judge(prompt, max_retries=3):
    global active_key_index
    # Reasoning disabled: the probe showed 0 reasoning tokens and they bill at
    # the completion rate. max_tokens is a backstop - the probe used 64.
    payload = {"model": V.JUDGE_MODEL,
               "messages": [{"role": "user", "content": prompt}],
               "temperature": 0.0, "max_tokens": 512,
               "reasoning": {"enabled": False}}
    for attempt in range(max_retries):
        for _ in range(len(api_keys)):
            with key_lock:
                k = api_keys[active_key_index]
            try:
                r = requests.post(
                    "https://openrouter.ai/api/v1/chat/completions",
                    headers={"Authorization": f"Bearer {k}", "Content-Type": "application/json"},
                    json=payload, timeout=120)
            except requests.RequestException:
                break
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"]
            if r.status_code in (401, 402, 403, 429):
                with key_lock:
                    active_key_index = (active_key_index + 1) % len(api_keys)
                continue
            break
        time.sleep(2 * (attempt + 1))
    return None


def run_job(job):
    raw = call_judge(job["prompt"])
    verdicts = V.parse_fact_verdicts(raw, job["n_facts"]) if raw else None
    ok, reason = V.accept(verdicts, job["roles"], job["operation"])
    targeted = ([v for v, r in zip(verdicts, job["roles"]) if r == "target"]
                if verdicts else [])
    return {
        "job_key": job["job_key"], "gen": job["gen"], "pipeline": job["pipeline"],
        "doc_id": job["doc_id"], "method": job["method"], "n": job["n"],
        "target_categories": job["target_categories"],
        "accepted": ok, "reason": reason,
        "target_verdicts": " | ".join(targeted),
        "raw": raw,
    }


print("Ready.")

Ready.


In [4]:
# PRE-FLIGHT. Two calls, about $0.002, before committing to 400.
smoke_ok = True
for op in ("delete", "alter"):
    probe = next(j for j in jobs if j["operation"] == op)
    raw = call_judge(probe["prompt"])
    verdicts = V.parse_fact_verdicts(raw, probe["n_facts"]) if raw else None
    if verdicts is None:
        smoke_ok = False
    print(f"{op:>7}  facts={probe['n_facts']}  parsed={verdicts is not None}  {verdicts}")

if not smoke_ok:
    raise RuntimeError("Pre-flight failed. Do not run the full pass.")
print("Pre-flight passed.")

 delete  facts=7  parsed=True  ['NOT_MENTIONED', 'NOT_MENTIONED', 'SUPPORTED', 'NOT_MENTIONED', 'SUPPORTED', 'SUPPORTED', 'SUPPORTED']


  alter  facts=7  parsed=True  ['SUPPORTED', 'CONTRADICTED', 'SUPPORTED', 'SUPPORTED', 'SUPPORTED', 'SUPPORTED', 'SUPPORTED']
Pre-flight passed.


In [5]:
done = {}
if os.path.exists(BACKUP_FILE):
    prior = pd.read_csv(BACKUP_FILE)
    done = {r["job_key"]: r for _, r in prior.iterrows()}
    print(f"Resuming: {len(done)} already verified.")
else:
    print("No backup - starting fresh.")

pending = [j for j in jobs if j["job_key"] not in done]
print(f"{len(pending)} of {len(jobs)} still to verify.")
print("")

start = time.time()
results = [done[j["job_key"]].to_dict() for j in jobs if j["job_key"] in done]
lock = threading.Lock()


def checkpoint():
    pd.DataFrame(results).to_csv(BACKUP_FILE, index=False, encoding="utf-8-sig")


if pending:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        for finished, record in enumerate(pool.map(run_job, pending), start=1):
            with lock:
                results.append(record)
                if finished % CHECKPOINT_EVERY == 0 or finished == len(pending):
                    checkpoint()
                    acc = sum(bool(r.get("accepted")) for r in results)
                    print(f"   --- saved at {finished}/{len(pending)} "
                          f"(total {len(results)}, accepted={acc}, {time.time() - start:.0f}s) ---")

checkpoint()
v = pd.DataFrame(results)
v["accepted"] = v["accepted"].astype(bool)
v.drop(columns=["raw"]).to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print("")
print(f"{len(v)} verified, {int(v['accepted'].sum())} accepted "
      f"({100 * v['accepted'].mean():.0f}%)")

No backup - starting fresh.
400 of 400 still to verify.



   --- saved at 20/400 (total 20, accepted=7, 10s) ---


   --- saved at 40/400 (total 40, accepted=13, 18s) ---


   --- saved at 60/400 (total 60, accepted=25, 27s) ---


   --- saved at 80/400 (total 80, accepted=33, 35s) ---


...

   --- saved at 320/400 (total 320, accepted=147, 129s) ---


   --- saved at 340/400 (total 340, accepted=151, 137s) ---


   --- saved at 360/400 (total 360, accepted=156, 145s) ---


   --- saved at 380/400 (total 380, accepted=165, 153s) ---


   --- saved at 400/400 (total 400, accepted=171, 161s) ---

400 verified, 171 accepted (43%)


In [6]:
print("ACCEPTANCE BY GENERATOR AND VARIANT\n")
v["variant"] = v.apply(
    lambda r: r["method"] if r["pipeline"] == "alteration" else f"Delete N={int(r['n'])}", axis=1)
t = v.groupby(["gen", "variant"]).agg(items=("accepted", "size"), accepted=("accepted", "sum"))
t["rate%"] = (100 * t["accepted"] / t["items"]).round(0)
print(t.to_string())

print("\n\nWHY ITEMS WERE REJECTED\n")
print(v[~v["accepted"]]["reason"].value_counts().to_string())

print("\n\nACCEPTANCE BY FIRST TARGET CATEGORY\n")
v["cat"] = v["target_categories"].astype(str).str.split(" | ", regex=False).str[0]
c = v.groupby("cat").agg(items=("accepted", "size"), accepted=("accepted", "sum"))
c["rate%"] = (100 * c["accepted"] / c["items"]).round(0)
print(c.sort_values("items", ascending=False).to_string())

ACCEPTANCE BY GENERATOR AND VARIANT

                     items  accepted  rate%
gen      variant                           
deepseek Coarsen        32         2    6.0
         Delete N=1     50        32   64.0
         Delete N=2     40        17   42.0
         Delete N=3     28        11   39.0
         Substitute     50        23   46.0
luna     Coarsen        32         1    3.0
         Delete N=1     48        29   60.0
         Delete N=2     40        16   40.0
         Delete N=3     30        13   43.0
         Substitute     50        27   54.0


WHY ITEMS WERE REJECTED

reason
collateral damage to an untargeted fact                    94
wrong operation applied, item belongs in the other pool    81
targeted fact still supported, the edit did not take       54


ACCEPTANCE BY FIRST TARGET CATEGORY

                  items  accepted  rate%
cat                                     
Temporal            123        37   30.0
Symptom              92        41   45.0
Age         